# Graph-Based Warehouse Slotting Optimization

**Objective**: Optimize warehouse layout by placing frequently co-purchased products closer together using Graph Theory and Node2Vec embeddings.

### Methodology Overview
1. **Bipartite Graph**: Model relationships between Orders and Products.
2. **Graph Projection**: Convert to a Product-Product graph weighted by Jaccard Similarity.
3. **Node2Vec**: Learn high-dimensional embeddings that capture the context of product co-occurrence.
4. **Zoning**: Use K-Means clustering to group products into "Zones".
5. **Simulation**: Compare picking distance (Total Manhattan Distance) of our Optimized layout vs. a Random Baseline.

In [ ]:
import pandas as pd
import networkx as nx
from networkx.algorithms import bipartite
import numpy as np
import random
import matplotlib.pyplot as plt
from gensim.models import Word2Vec
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
import community as community_louvain
import os

# Setting seed for reproducibility
random.seed(42)
np.random.seed(42)

## 1. Data Loading
We utilize the Olist Brazilian E-Commerce dataset. Specifically, we need the `order_items` to link orders to products, and `products` for metadata.

In [ ]:
data_dir = 'data'
order_items = pd.read_csv(os.path.join(data_dir, 'olist_order_items_dataset.csv'))
products = pd.read_csv(os.path.join(data_dir, 'olist_products_dataset.csv'))

print(f"Loaded {len(order_items)} order items and {len(products)} unique products.")
order_items.head()

## 2. Graph Construction
### Bipartite Graph (Orders <-> Products)
Nodes are of two types: `order_id` (0) and `product_id` (1). An edge exists if a product is part of an order.

In [ ]:
# To keep the graph manageable for the demo, we sample orders if needed
# For this run, we'll try to build the full graph but filter out products with extremely low volume
product_counts = order_items['product_id'].value_counts()
stable_products = product_counts[product_counts > 1].index # Denoising: focus on products bought more than once
filtered_items = order_items[order_items['product_id'].isin(stable_products)]

B = nx.Graph()
B.add_nodes_from(filtered_items['order_id'].unique(), bipartite=0)
B.add_nodes_from(filtered_items['product_id'].unique(), bipartite=1)

edges = list(zip(filtered_items['order_id'], filtered_items['product_id']))
B.add_edges_from(edges)

print(f"Bipartite Graph: {len(B.nodes)} nodes, {len(B.edges)} edges.")

### Product-Product Projection
We project the bipartite graph onto product nodes. Two products are connected if they have appeared together in at least one order. We weight the edges using **Jaccard Similarity** to handle popularity bias.

In [ ]:
product_nodes = [n for n, d in B.nodes(data=True) if d['bipartite'] == 1]

# Projecting (this creates edges between products that share an order)
print("Projecting graph (this may take a minute color)...")
P = bipartite.weighted_projected_graph(B, product_nodes)

# Apply Jaccard Similarity weighting
for u, v, d in P.edges(data=True):
    intersection = d['weight'] # Shared orders count
    union = B.degree(u) + B.degree(v) - intersection
    d['weight'] = intersection / union if union > 0 else 0

print(f"Product Graph: {len(P.nodes)} nodes, {len(P.edges)} edges.")

## 3. Node2Vec Embedding Generation
We use biased random walks to learn the structural neighborhood of products. This identifies latent communities of products that are functionally related even if they don't share every order.

In [ ]:
def get_biased_walks(G, walk_length=20, num_walks=10):
    walks = []
    nodes = list(G.nodes())
    for _ in range(num_walks):
        random.shuffle(nodes)
        for node in nodes:
            walk = [node]
            while len(walk) < walk_length:
                curr = walk[-1]
                nbrs = list(G.neighbors(curr))
                if nbrs:
                    # Simplification: Uniform random walk on weighted graph
                    # Node2Vec p/q bias can be added for deeper exploration
                    walk.append(random.choice(nbrs))
                else:
                    break
            walks.append([str(node) for node in walk])
    return walks

print("Generating random walks...")
walks = get_biased_walks(P)

print("Training Word2Vec model...")
model = Word2Vec(walks, vector_size=64, window=5, min_count=1, sg=1, workers=4)

embeddings = {str(node): model.wv[str(node)] for node in P.nodes()}
print("Embeddings generated.")

## 4. Clustering into Storage Zones
We group product vectors into 12 distinct "Zones" for the warehouse.

In [ ]:
product_ids = list(embeddings.keys())
vectors = np.array([embeddings[pid] for pid in product_ids])

n_zones = 12
kmeans = KMeans(n_clusters=n_zones, random_state=42, n_init=10)
zones = kmeans.fit_predict(vectors)
product_zones = {pid: zone for pid, zone in zip(product_ids, zones)}

print(f"Products grouped into {n_zones} zones.")

## 5. Warehouse Layout & Simulation
We simulate a grid-based warehouse. 
- **Baseline**: Products are placed randomly throughout the grid.
- **Optimized**: Products are grouped by their derived Zone ID.

In [ ]:
class WarehouseSim:
    def __init__(self, pids, zones):
        self.pids = pids
        self.zones = zones
        self.grid_side = int(np.ceil(np.sqrt(len(pids)))) + 2
        
    def get_baseline_layout(self):
        locs = [(x, y) for x in range(self.grid_side) for y in range(self.grid_side)]
        random.shuffle(locs)
        return {pid: locs[i] for i, pid in enumerate(self.pids)}
    
    def get_optimized_layout(self):
        sorted_pids = sorted(self.pids, key=lambda x: self.zones[x])
        layout = {}
        for i, pid in enumerate(sorted_pids):
            x = i % self.grid_side
            y = i // self.grid_side
            layout[pid] = (x, y)
        return layout

    @staticmethod
    def get_dist(p1, p2):
        return abs(p1[0]-p2[0]) + abs(p1[1]-p2[1])
    
    def simulate_picking(self, orders, layout, samples=1000):
        total_dist = 0
        count = 0
        valid_orders = [o for o in orders if len([p for p in o if p in layout]) > 0]
        sampled_orders = random.sample(valid_orders, min(samples, len(valid_orders)))
        
        for order in sampled_orders:
            dist = 0
            curr = (0, 0)
            for pid in order:
                if pid in layout:
                    target = layout[pid]
                    dist += self.get_dist(curr, target)
                    curr = target
            dist += self.get_dist(curr, (0, 0)) # Return to start
            total_dist += dist
            count += 1
        return total_dist / count if count > 0 else 0

sim = WarehouseSim(product_ids, product_zones)
baseline_layout = sim.get_baseline_layout()
optimized_layout = sim.get_optimized_layout()

all_orders = order_items.groupby('order_id')['product_id'].apply(list).tolist()

print("Running Simulation...")
base_avg = sim.simulate_picking(all_orders, baseline_layout)
opt_avg = sim.simulate_picking(all_orders, optimized_layout)

improvement = (base_avg - opt_avg) / base_avg * 100
print(f"\nBaseline Avg Distance: {base_avg:.2f}")
print(f"Optimized Avg Distance: {opt_avg:.2f}")
print(f"Layout Efficiency Improvement: {improvement:.1f}%")

## 6. Visualization
### Warehouse Layout Map
Dots represent products, colored by their assigned Zone ID.

In [ ]:
def plot_grid(layout, title):
    x = [l[0] for l in layout.values()]
    y = [l[1] for l in layout.values()]
    c = [product_zones[p] for p in layout.keys()]
    
    plt.figure(figsize=(10, 8))
    plt.scatter(x, y, c=c, cmap='tab20', s=10)
    plt.title(title)
    plt.colorbar(label='Zone ID')
    plt.show()

plot_grid(optimized_layout, "Optimized Slotting: Grouped by Product Affinity")

### Performance Metric
Comparison of picking distances.

In [ ]:
plt.figure(figsize=(6, 5))
plt.bar(['Baseline', 'Optimized'], [base_avg, opt_avg], color=['#bbbbbb', '#5dade2'])
plt.ylabel('Avg Manhattan Distance per Order')
plt.title('Optimization Result: Distance Reduction')
plt.show()

### Cluster Embeddings (t-SNE)
Visualization of high-dimensional product vectors projected to 2D.

In [ ]:
print("Projecting embeddings with t-SNE...")
tsne = TSNE(n_components=2, perplexity=30, max_iter=300, random_state=42)
reduced = tsne.fit_transform(vectors)

plt.figure(figsize=(10, 8))
plt.scatter(reduced[:, 0], reduced[:, 1], c=zones, cmap='tab20', s=5, alpha=0.5)
plt.title("Product neighborhood Clusters (t-SNE)")
plt.show()